# Bootstrap GMM Analysis

This notebook configures and launches `bootstrap_GMM.py` for bootstrap Gaussian mixture modeling (GMM) of geochronologic age distributions.

For routine use, modify only the parameters in **Section 1: User configuration**. The remaining sections validate the settings, pass them to the Python script through environment variables, control parallel execution, and launch the complete analysis.

The input Excel workbook should contain the columns required by `bootstrap_GMM.py`, including `Sample_ID` and `BestAge`.


In [ ]:
import sys

# Display the Python interpreter used by the current Jupyter kernel.
# This is useful for confirming that the notebook is running in the
# intended Conda environment (e.g., gmm_openblas).
print("Python executable:")
print(sys.executable)


In [ ]:
import numpy as np
import pandas as pd
import sklearn
import joblib
import matplotlib
import openpyxl

from sklearn.mixture import GaussianMixture

# Report the main package versions used by the analysis.
# Recording these versions helps document the computational environment
# used for reproducibility.
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)
print("Matplotlib:", matplotlib.__version__)
print("openpyxl:", openpyxl.__version__)
print("All required dependencies imported successfully.")


In [ ]:
import os
import sys
import subprocess
from pathlib import Path


# ============================================================
# 1. USER CONFIGURATION
#    For routine analyses, modify only the settings in this section.
# ============================================================

# Path to the main Python script that performs the complete bootstrap GMM
# analysis. The filename is aligned with the GitHub file: bootstrap_GMM.py.
script_path = Path(
    "/media/tigerwp/data/Xiehongsen/Illustrate of Yangtze data/"
    "bootstrap GMM/bootstrap_GMM.py"
)

# Path to the Excel workbook containing the geochronologic age data.
# The Python script expects, at minimum, the columns "Sample_ID" and "BestAge".
excel_path = Path(
    "/media/tigerwp/data/Xiehongsen/Illustrate of Yangtze data/"
    "baserock python data-1.xlsx"
)

# Directory in which all numerical outputs and figures will be saved.
# The directory will be created automatically if it does not already exist.
output_dir = Path(
    "/media/tigerwp/data/Xiehongsen/Illustrate of Yangtze data/"
    "bootstrap GMM/new figure 60/"
)

# Sample_ID values for the two datasets to be analyzed.
# These strings must exactly match entries in the "Sample_ID" column
# of the input Excel workbook.
sample_a = "Zr-Lower Yangtze"
sample_b = "Ar-Lower Yangtze"

# Prefix used for all output files generated by bootstrap_GMM.py.
# Do not include a file extension such as .png, .csv, or .xlsx.
output_name = "Lower Yangtze-GMM"

# Number of bootstrap replicates.
# The final analysis uses 5000 replicates. A smaller value can be used
# temporarily for testing the workflow.
n_bootstrap = 5000

# Number of parallel worker processes used by joblib.
# Adjust this value according to the CPU resources available on the machine
# or computing server.
n_jobs = 64

# Number of independent initializations used for each GaussianMixture fit.
# Multiple initializations reduce sensitivity to local optima in the
# expectation-maximization optimization.
gmm_n_init = 10


# ============================================================
# 2. FILE AND PARAMETER VALIDATION
#    Check the configuration before starting the expensive calculation.
# ============================================================

# Confirm that the main Python analysis script exists.
if not script_path.is_file():
    raise FileNotFoundError(
        f"Python analysis script not found:\n{script_path}"
    )

# Confirm that the input Excel workbook exists.
if not excel_path.is_file():
    raise FileNotFoundError(
        f"Input Excel workbook not found:\n{excel_path}"
    )

# Both sample identifiers must be non-empty strings.
if not sample_a.strip():
    raise ValueError("sample_a cannot be empty.")

if not sample_b.strip():
    raise ValueError("sample_b cannot be empty.")

# The two selected datasets must be different.
if sample_a.strip() == sample_b.strip():
    raise ValueError("sample_a and sample_b must be different.")

# At least one bootstrap replicate is required.
if n_bootstrap < 1:
    raise ValueError("n_bootstrap must be greater than 0.")

# joblib does not accept zero workers.
if n_jobs == 0:
    raise ValueError("n_jobs cannot be 0.")

# GaussianMixture requires at least one initialization.
if gmm_n_init < 1:
    raise ValueError("gmm_n_init must be greater than 0.")

# Create the output directory, including any missing parent directories.
output_dir.mkdir(parents=True, exist_ok=True)


# ============================================================
# 3. PASS THE CONFIGURATION TO bootstrap_GMM.py
#    The main script reads these settings from environment variables.
# ============================================================

# Start from a copy of the current process environment so that the subprocess
# inherits the active Python/Conda environment and other system settings.
env = os.environ.copy()

# Input/output paths, sample identifiers, and output filename prefix.
env["GMM_EXCEL_PATH"] = str(excel_path)
env["GMM_OUT_DIR"] = str(output_dir)
env["GMM_SAMPLE_A"] = sample_a.strip()
env["GMM_SAMPLE_B"] = sample_b.strip()
env["GMM_FNAME"] = output_name.strip()

# Bootstrap and Gaussian mixture model settings.
# Environment-variable values are strings, so numeric settings are converted
# explicitly before being passed to the subprocess.
env["GMM_N_BOOTSTRAP"] = str(n_bootstrap)
env["GMM_N_JOBS"] = str(n_jobs)
env["GMM_N_INIT"] = str(gmm_n_init)

# Restrict low-level numerical libraries to one thread per worker process.
# Bootstrap replicates are parallelized by joblib at the process level.
# Without these limits, each worker could start additional BLAS/OpenMP
# threads, causing nested parallelism, CPU oversubscription, and reduced
# performance on multicore systems.
env["OMP_NUM_THREADS"] = "1"
env["MKL_NUM_THREADS"] = "1"
env["OPENBLAS_NUM_THREADS"] = "1"
env["NUMEXPR_NUM_THREADS"] = "1"
env["VECLIB_MAXIMUM_THREADS"] = "1"
env["BLIS_NUM_THREADS"] = "1"

# Use a non-interactive Matplotlib backend. This prevents graphical windows
# from blocking execution on remote servers or headless Linux systems while
# still allowing figures to be written to disk.
env["MPLBACKEND"] = "Agg"


# ============================================================
# 4. DISPLAY THE CURRENT RUN CONFIGURATION
#    This provides a concise record of the settings used for the analysis.
# ============================================================

print("Bootstrap GMM run configuration")
print("-" * 70)
print("Python executable :", sys.executable)
print("Analysis script   :", script_path)
print("Input Excel file  :", excel_path)
print("Output directory  :", output_dir)
print("Sample A          :", sample_a)
print("Sample B          :", sample_b)
print("Output prefix     :", output_name)
print("Bootstrap runs    :", n_bootstrap)
print("Parallel workers  :", n_jobs)
print("GMM n_init        :", gmm_n_init)
print("-" * 70)


# ============================================================
# 5. RUN THE COMPLETE bootstrap_GMM.py ANALYSIS
# ============================================================

# Use sys.executable so that bootstrap_GMM.py is launched with the same
# Python interpreter and Conda environment as the current Jupyter kernel.
#
# check=False is used so that the return code can be inspected explicitly
# after execution. All stdout/stderr from bootstrap_GMM.py remains visible
# in the notebook while the calculation is running.
result = subprocess.run(
    [sys.executable, str(script_path)],
    env=env,
    check=False
)

print("-" * 70)
print("Process return code:", result.returncode)

# A return code of 0 conventionally indicates successful completion.
if result.returncode == 0:
    print("Bootstrap GMM analysis completed successfully.")
    print("Results saved to:")
    print(output_dir)
else:
    print("Bootstrap GMM analysis failed.")
    print("Please inspect the error messages printed above for details.")
